In [ ]:
# =============================================================================
# CELDA 0: INSTALACIÓN DE LIBRERÍAS NECESARIAS
# Entorno: Google Colab con GPU T4
# Ejecutar UNA sola vez al inicio de la sesión
# =============================================================================

# Librerías base de transformers e imágenes médicas
!pip install -q transformers==4.44.2          # Hugging Face: ViT, Swin, BEiT
!pip install -q timm==1.0.9                   # Modelos pre-entrenados de visión
# MONAI >=1.4 corrige el OverflowError de set_random_state con NumPy 2.x
!pip install -q "monai>=1.4.0"                # Imagen médica (Swin UNETR)
!pip install -q einops==0.8.0                 # Reorganización de tensores
!pip install -q nibabel==5.2.1                # Lectura de NIfTI (.nii / .nii.gz)
!pip install -q SimpleITK==2.4.0              # Procesamiento de imagen médica

# Librerías de explicabilidad y métricas
!pip install -q shap==0.46.0                  # SHAP values
!pip install -q captum==0.7.0                 # Atribuciones (Integrated Grad, etc.)
!pip install -q quantus==0.5.3                # Métricas de XAI estandarizadas
!pip install -q grad-cam==1.5.4               # Mapas de activación

# Métricas clínicas y utilidades
# Reemplazar versión fija por una >=1.6 para evitar conflictos con hdbscan y umap-learn
!pip install -q "scikit-learn>=1.6,<1.7"      # AUC, F1, sensibilidad, especificidad
!pip install -q medpy==0.5.2                  # HD95, métricas de segmentación
!pip install -q seaborn matplotlib opencv-python tqdm pandas

# Verificación de GPU T4
import torch
print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU")
print("PyTorch:", torch.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 MB 13.8 MB/s eta 0:00:00


In [ ]:
# =============================================================================
# CELDA 1: CREACIÓN DE CARPETAS PARA CADA TRANSFORMER
# Cada transformer tiene su propia carpeta con un tipo de imagen específico
# =============================================================================
import os
from pathlib import Path

# Directorio raíz dentro del entorno de Colab
ROOT = Path("/content/transformers_medicos")
ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Definición de carpetas y descripción de las imágenes que cada modelo necesita
# -----------------------------------------------------------------------------
CARPETAS = {
    "imagenes_vit":         "Radiografías de tórax 2D (.png/.jpg) 224x224 RGB. "
                            "Ej: NIH ChestX-ray14, RSNA Pneumonia. "
                            "Tarea: clasificación (neumonía sí/no, COVID, etc.)",

    "imagenes_swin":        "Imágenes histopatológicas 2D (.png/.tiff) 224x224 RGB. "
                            "Ej: PatchCamelyon, BreakHis, parches WSI de mama/colon. "
                            "Tarea: clasificación tumor benigno/maligno.",

    "imagenes_beit":        "Imágenes dermatoscópicas 2D (.jpg) 224x224 RGB. "
                            "Ej: ISIC 2019/2020, HAM10000 (lesiones de piel). "
                            "Tarea: clasificación melanoma vs nevo.",

    "imagenes_swin_unetr":  "Volúmenes 3D en formato NIfTI (.nii.gz). "
                            "Ej: BraTS (resonancia cerebral T1/T1ce/T2/FLAIR), "
                            "BTCV (TC abdominal multi-órgano). "
                            "Tarea: segmentación volumétrica 3D.",

    "imagenes_transunet":   "Cortes 2D de TC/RM (.png o slices de NIfTI) 224x224. "
                            "Ej: Synapse multi-organ, ACDC cardíaco. "
                            "Tarea: segmentación 2D multi-clase.",

    "imagenes_swin_unet":   "Cortes 2D de RM cardíaca (.png) 224x224 escala de grises. "
                            "Ej: ACDC, M&Ms (ventrículos y miocardio). "
                            "Tarea: segmentación 2D de estructuras cardíacas.",
}

# Crear cada subcarpeta y un README con la descripción
for nombre, descripcion in CARPETAS.items():
    carpeta = ROOT / nombre
    carpeta.mkdir(exist_ok=True)
    # README explicando qué imágenes colocar en cada carpeta
    with open(carpeta / "README.txt", "w", encoding="utf-8") as f:
        f.write(f"Carpeta: {nombre}\n")
        f.write(f"Tipo de imagen esperada:\n{descripcion}\n")
    print(f"✅ {carpeta}  ->  {descripcion[:70]}...")

# Configuración global compartida
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED)

In [ ]:
# =============================================================================
# CELDA 12: DESCARGA AUTOMÁTICA DE DATASETS PÚBLICOS MÉDICOS
# Datasets cubiertos:
#   - ChestX-ray14   -> carpeta imagenes_vit       (clasificación radiografías)
#   - ISIC 2019      -> carpeta imagenes_beit      (clasificación dermatoscopía)
#   - PatchCamelyon  -> carpeta imagenes_swin      (clasificación histopatología)
#   - BraTS 2021     -> carpeta imagenes_swin_unetr (segmentación 3D cerebral)
#   - ACDC           -> carpeta imagenes_swin_unet (segmentación 2D cardíaca)
#   - Synapse        -> carpeta imagenes_transunet (segmentación 2D abdominal)
#
# Fuentes: Hugging Face Datasets + Kaggle API + repositorios oficiales
# Entorno: Google Colab con GPU T4
# =============================================================================

# -----------------------------------------------------------------------------
# 12.0  INSTALACIÓN DE LIBRERÍAS NECESARIAS PARA DESCARGA
# -----------------------------------------------------------------------------
# Versiones compatibles con gradio, diffusers, peft y gcsfs ya instalados en Colab
!pip install -q "datasets>=3.0.0,<4.0.0"        # Hugging Face Datasets (compat. con HF Hub nuevo)
!pip install -q "kaggle>=1.6.17"                # API oficial de Kaggle
!pip install -q "gdown>=5.2.0"                  # Descarga desde Google Drive público
!pip install -q "huggingface_hub>=0.34.0,<1.0"  # Compatible con gradio/diffusers/peft
!pip install -q "fsspec>=2025.3.0"              # Compatible con gcsfs 2025.3.0

import os
import shutil
import zipfile
import tarfile
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image

ROOT = Path("/content/transformers_medicos")
ROOT.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 12.1  CONFIGURACIÓN DE CREDENCIALES KAGGLE
# Necesitas tu archivo kaggle.json (Cuenta Kaggle -> Settings -> Create API Token)
# Súbelo manualmente o monta Drive. Aquí mostramos las dos opciones.
# =============================================================================
def configurar_kaggle():
    """Configura la API de Kaggle copiando kaggle.json al directorio ~/.kaggle/"""
    from google.colab import files
    print("📤 Sube tu archivo kaggle.json (Kaggle -> Account -> Create New API Token)")
    # Subida manual del token
    uploaded = files.upload()
    # Mover el archivo al lugar esperado por la API
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.move("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    # La API exige permisos 600 sobre el token
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle API configurada correctamente")

# Alternativa: leer kaggle.json desde Google Drive ya montado
def configurar_kaggle_desde_drive(ruta_drive="/content/drive/MyDrive/kaggle.json"):
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.copy(ruta_drive, os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle API configurada desde Drive")

# =============================================================================
# 12.2  CHESTX-RAY14 (Hugging Face: alkzar90/NIH-Chest-X-ray-dataset)
# Va a la carpeta imagenes_vit como clasificación binaria normal vs patológico
# =============================================================================
def descargar_chestxray14(n_muestras=2000):
    """
    Descarga un subconjunto del NIH ChestX-ray14 desde Hugging Face.
    n_muestras: cuántas imágenes bajar (el dataset completo tiene ~112k, ~45 GB).
    """
    from datasets import load_dataset

    destino = ROOT / "imagenes_vit"
    (destino / "normal").mkdir(parents=True, exist_ok=True)
    (destino / "patologico").mkdir(parents=True, exist_ok=True)

    print("⬇️  Descargando ChestX-ray14 desde Hugging Face...")
    # Streaming para no descargar todo el dataset (45 GB completos)
    ds = load_dataset(
        "alkzar90/NIH-Chest-X-ray-dataset",
        "image-classification",
        split="train",
        streaming=True,                    # Iterador lazy: ahorra disco/RAM
        trust_remote_code=True,
    )

    # NIH ChestX-ray14 tiene 14 etiquetas + "No Finding"
    # Reagrupamos en binario: "No Finding" = normal, resto = patológico
    contador = {"normal": 0, "patologico": 0}
    objetivo = n_muestras // 2

    for i, ejemplo in enumerate(tqdm(ds, total=n_muestras)):
        if contador["normal"] >= objetivo and contador["patologico"] >= objetivo:
            break
        etiquetas = ejemplo["labels"]                       # Lista de strings
        clase = "normal" if "No Finding" in etiquetas else "patologico"
        if contador[clase] >= objetivo:
            continue
        # La imagen viene como PIL.Image; la guardamos como PNG 224x224
        img = ejemplo["image"].convert("RGB").resize((224, 224))
        img.save(destino / clase / f"chest_{contador[clase]:05d}.png")
        contador[clase] += 1

    print(f"✅ ChestX-ray14: normal={contador['normal']}, "
          f"patológico={contador['patologico']}")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.3  ISIC 2019 (Hugging Face: marmal88/skin_cancer)
# Va a la carpeta imagenes_beit como clasificación benigno vs maligno
# =============================================================================
def descargar_isic(n_muestras=3000):
    """
    Descarga ISIC (skin cancer) desde Hugging Face y lo organiza en
    imagenes_beit/benigno y imagenes_beit/maligno.
    """
    from datasets import load_dataset

    destino = ROOT / "imagenes_beit"
    (destino / "benigno").mkdir(parents=True, exist_ok=True)
    (destino / "maligno").mkdir(parents=True, exist_ok=True)

    print("⬇️  Descargando ISIC skin cancer desde Hugging Face...")
    ds = load_dataset("marmal88/skin_cancer", split="train", streaming=True)

    # Etiquetas malignas reportadas en ISIC: melanoma, basal cell carcinoma, etc.
    etiq_maligno = {"melanoma", "basal_cell_carcinoma", "actinic_keratoses",
                    "squamous_cell_carcinoma"}

    contador = {"benigno": 0, "maligno": 0}
    objetivo = n_muestras // 2

    for ejemplo in tqdm(ds, total=n_muestras):
        if all(c >= objetivo for c in contador.values()):
            break
        dx = ejemplo.get("dx", "").lower()
        clase = "maligno" if dx in etiq_maligno else "benigno"
        if contador[clase] >= objetivo:
            continue
        img = ejemplo["image"].convert("RGB").resize((224, 224))
        img.save(destino / clase / f"isic_{contador[clase]:05d}.jpg",
                 quality=92)
        contador[clase] += 1

    print(f"✅ ISIC: benigno={contador['benigno']}, maligno={contador['maligno']}")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.4  PATCHCAMELYON (Hugging Face: 1aurent/PatchCamelyon)
# Va a la carpeta imagenes_swin como clasificación tumor sí/no en histopatología
# =============================================================================
def descargar_patchcamelyon(n_muestras=4000):
    """Parches 96x96 de WSI de mama. Lo redimensionamos a 224x224 para Swin."""
    from datasets import load_dataset

    destino = ROOT / "imagenes_swin"
    (destino / "tumor_no").mkdir(parents=True, exist_ok=True)
    (destino / "tumor_si").mkdir(parents=True, exist_ok=True)

    print("⬇️  Descargando PatchCamelyon desde Hugging Face...")
    ds = load_dataset("1aurent/PatchCamelyon", split="train", streaming=True)

    contador = {"tumor_no": 0, "tumor_si": 0}
    objetivo = n_muestras // 2

    for ejemplo in tqdm(ds, total=n_muestras):
        if all(c >= objetivo for c in contador.values()):
            break
        # 0 = no tumor, 1 = tumor
        clase = "tumor_si" if ejemplo["label"] == 1 else "tumor_no"
        if contador[clase] >= objetivo:
            continue
        img = ejemplo["image"].convert("RGB").resize((224, 224))
        img.save(destino / clase / f"pcam_{contador[clase]:05d}.png")
        contador[clase] += 1

    print(f"✅ PatchCamelyon: no_tumor={contador['tumor_no']}, "
          f"tumor={contador['tumor_si']}")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.5  BRATS 2021 (Kaggle: dschettler8845/brats-2021-task1)
# Va a la carpeta imagenes_swin_unetr (segmentación 3D)
# Estructura final: imagenesTr/*.nii.gz y etiquetasTr/*.nii.gz
# =============================================================================
def descargar_brats(n_pacientes=10):
    """
    Descarga un subconjunto de BraTS 2021 desde Kaggle.
    Cada paciente trae 4 modalidades (T1, T1ce, T2, FLAIR) + segmentación.
    Usamos solo FLAIR como canal único para Swin UNETR (in_channels=1).
    """
    destino = ROOT / "imagenes_swin_unetr"
    (destino / "imagenesTr").mkdir(parents=True, exist_ok=True)
    (destino / "etiquetasTr").mkdir(parents=True, exist_ok=True)
    tmp = ROOT / "_tmp_brats"
    tmp.mkdir(exist_ok=True)

    print("⬇️  Descargando BraTS 2021 desde Kaggle (~12 GB completo)...")
    # Descarga vía Kaggle API; -p define la carpeta destino, --unzip extrae
    os.system(
        f"kaggle datasets download -d dschettler8845/brats-2021-task1 "
        f"-p {tmp} --unzip"
    )

    # BraTS21 viene en ZIPs por paciente; recorremos los primeros N
    import glob
    pacientes = sorted(glob.glob(str(tmp / "BraTS2021_*")))[:n_pacientes]

    for p in tqdm(pacientes):
        pid = Path(p).name
        # Copiar FLAIR como imagen y la segmentación como etiqueta
        flair = Path(p) / f"{pid}_flair.nii.gz"
        seg = Path(p) / f"{pid}_seg.nii.gz"
        if flair.exists() and seg.exists():
            shutil.copy(flair, destino / "imagenesTr" / f"{pid}.nii.gz")
            shutil.copy(seg, destino / "etiquetasTr" / f"{pid}.nii.gz")

    # Limpieza para no llenar el disco de Colab
    shutil.rmtree(tmp, ignore_errors=True)

    n_imgs = len(list((destino / "imagenesTr").glob("*.nii.gz")))
    print(f"✅ BraTS: {n_imgs} pacientes guardados")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.6  ACDC (Kaggle: jeffyo/automated-cardiac-diagnosis-challenge-acdc)
# Va a la carpeta imagenes_swin_unet (segmentación 2D cardíaca)
# Convertimos los NIfTI 3D a cortes 2D PNG para entrenar el Swin-Unet 2D
# =============================================================================
def descargar_acdc(n_pacientes=20):
    """Descarga ACDC y extrae cortes 2D (RM cardíaca cine)."""
    import nibabel as nib
    import glob

    destino = ROOT / "imagenes_swin_unet"
    (destino / "imagenes").mkdir(parents=True, exist_ok=True)
    (destino / "mascaras").mkdir(parents=True, exist_ok=True)
    tmp = ROOT / "_tmp_acdc"
    tmp.mkdir(exist_ok=True)

    print("⬇️  Descargando ACDC desde Kaggle...")
    os.system(
        f"kaggle datasets download -d jeffyo/automated-cardiac-diagnosis-challenge-acdc "
        f"-p {tmp} --unzip"
    )

    # Buscar pacientes en la estructura típica de ACDC
    pacientes = sorted(glob.glob(str(tmp / "**" / "patient*"), recursive=True))[:n_pacientes]

    cortes_guardados = 0
    for p in tqdm(pacientes):
        # ACDC trae frame ED (end-diastole) y ES (end-systole) con su _gt
        for fase in ["frame01", "frame12"]:                  # Aproximaciones ED/ES
            img_paths = glob.glob(str(Path(p) / f"*{fase}.nii.gz"))
            gt_paths = glob.glob(str(Path(p) / f"*{fase}_gt.nii.gz"))
            if not img_paths or not gt_paths:
                continue
            vol = nib.load(img_paths[0]).get_fdata()         # [H, W, D]
            seg = nib.load(gt_paths[0]).get_fdata()
            # Recorrer cortes axiales y guardar como PNG 224x224
            for z in range(vol.shape[2]):
                corte_img = vol[:, :, z]
                corte_seg = seg[:, :, z].astype(np.uint8)
                # Saltar cortes vacíos (sin segmentación)
                if corte_seg.sum() == 0:
                    continue
                # Normalizar imagen a 0-255
                ci = corte_img - corte_img.min()
                ci = (ci / (ci.max() + 1e-8) * 255).astype(np.uint8)
                Image.fromarray(ci).resize((224, 224)).save(
                    destino / "imagenes" / f"acdc_{cortes_guardados:05d}.png")
                Image.fromarray(corte_seg).resize((224, 224), Image.NEAREST).save(
                    destino / "mascaras" / f"acdc_{cortes_guardados:05d}.png")
                cortes_guardados += 1

    shutil.rmtree(tmp, ignore_errors=True)
    print(f"✅ ACDC: {cortes_guardados} cortes 2D guardados")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.7  SYNAPSE MULTI-ORGAN (TransUNet original)
# Va a la carpeta imagenes_transunet
# El dataset oficial requiere registro; se distribuye preprocesado vía Google Drive
# por los autores de TransUNet (https://github.com/Beckschen/TransUNet)
# =============================================================================
def descargar_synapse():
    """
    Descarga la versión preprocesada de Synapse (cortes 2D + máscaras)
    publicada por los autores de TransUNet a través de Google Drive.
    """
    import gdown

    destino = ROOT / "imagenes_transunet"
    (destino / "imagenes").mkdir(parents=True, exist_ok=True)
    (destino / "mascaras").mkdir(parents=True, exist_ok=True)
    tmp = ROOT / "_tmp_synapse"
    tmp.mkdir(exist_ok=True)

    # ID público de Drive con la versión preprocesada (archivo Synapse.zip)
    # Fuente: README oficial de TransUNet
    url = "https://drive.google.com/uc?id=18I9JHH_i0G5vTiVDKi7BBb4qNpMpNyyy"
    archivo = tmp / "Synapse.zip"

    print("⬇️  Descargando Synapse multi-organ (TransUNet preprocesado)...")
    gdown.download(url, str(archivo), quiet=False)

    # Extraer el ZIP
    with zipfile.ZipFile(archivo, "r") as z:
        z.extractall(tmp)

    # Convertir los .npz (formato del repo TransUNet) a PNG 224x224
    import glob
    npz_paths = sorted(glob.glob(str(tmp / "**" / "*.npz"), recursive=True))
    for i, npz_p in enumerate(tqdm(npz_paths)):
        d = np.load(npz_p)
        img = d["image"]; lbl = d["label"]
        # Normalizar imagen
        ci = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        ci = ci.astype(np.uint8)
        Image.fromarray(ci).resize((224, 224)).save(
            destino / "imagenes" / f"synapse_{i:05d}.png")
        Image.fromarray(lbl.astype(np.uint8)).resize((224, 224), Image.NEAREST).save(
            destino / "mascaras" / f"synapse_{i:05d}.png")

    shutil.rmtree(tmp, ignore_errors=True)
    print(f"✅ Synapse: {len(npz_paths)} cortes guardados")
    print(f"📂 Guardado en: {destino}")

# =============================================================================
# 12.8  ORQUESTADOR: descarga todos los datasets de una sola vez
# =============================================================================
def descargar_todos(
        n_chest=2000, n_isic=3000, n_pcam=4000,
        n_brats=10, n_acdc=20, incluir_synapse=True
):
    """
    Llama a todas las funciones anteriores en orden.
    Asegúrate de configurar Kaggle antes (configurar_kaggle() o desde Drive).
    """
    print("\n========== 1/6  ChestX-ray14 -> ViT ==========")
    descargar_chestxray14(n_muestras=n_chest)

    print("\n========== 2/6  ISIC -> BEiT ==========")
    descargar_isic(n_muestras=n_isic)

    print("\n========== 3/6  PatchCamelyon -> Swin ==========")
    descargar_patchcamelyon(n_muestras=n_pcam)

    print("\n========== 4/6  BraTS -> Swin UNETR ==========")
    descargar_brats(n_pacientes=n_brats)

    print("\n========== 5/6  ACDC -> Swin-Unet ==========")
    descargar_acdc(n_pacientes=n_acdc)

    if incluir_synapse:
        print("\n========== 6/6  Synapse -> TransUNet ==========")
        descargar_synapse()

    print("\n🎉 Descarga completa. Estructura final:")
    for sub in sorted(ROOT.iterdir()):
        if sub.is_dir():
            n = sum(1 for _ in sub.rglob("*") if _.is_file())
            print(f"   {sub.name:30s} -> {n} archivos")

# -----------------------------------------------------------------------------
# USO TÍPICO (descomenta lo que necesites):
# -----------------------------------------------------------------------------
# Paso 1: configurar Kaggle (solo necesario para BraTS y ACDC)
# configurar_kaggle()                       # Sube kaggle.json manualmente
# configurar_kaggle_desde_drive()           # O cárgalo desde Google Drive
#
# Paso 2: descargar todos los datasets
#descargar_todos(n_chest=1000, n_isic=1500, n_pcam=2000,
#                 n_brats=5, n_acdc=10, incluir_synapse=True)
#
# O descargar uno solo:
# descargar_chestxray14(n_muestras=1000)
# descargar_isic(n_muestras=1500)
# descargar_patchcamelyon(n_muestras=2000)
# descargar_brats(n_pacientes=5)
# descargar_acdc(n_pacientes=10)
# descargar_synapse()
# Descarga SOLO desde Hugging Face + Drive (no requieren Kaggle)
# Si quieres BraTS y ACDC, configura Kaggle primero con configurar_kaggle()
# y descomenta las dos últimas líneas.
descargar_chestxray14(n_muestras=400)     # -> imagenes_vit
descargar_isic(n_muestras=400)            # -> imagenes_beit
descargar_patchcamelyon(n_muestras=400)   # -> imagenes_swin
descargar_synapse()                       # -> imagenes_transunet
# configurar_kaggle()                     # SOLO si quieres BraTS y ACDC
# descargar_brats(n_pacientes=3)          # -> imagenes_swin_unetr
# descargar_acdc(n_pacientes=5)           # -> imagenes_swin_unet

In [ ]:
# =============================================================================
# CELDA 2: VISION TRANSFORMER (ViT)
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_vit
# Tipo de imagen:       Radiografías de tórax 2D (.png/.jpg), 224x224 RGB
# Tarea:                Clasificación binaria/multiclase (ej: neumonía sí/no)
# Modelo base:          google/vit-base-patch16-224
# =============================================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification
from pathlib import Path

# -----------------------------------------------------------------------------
# Dataset personalizado para leer imágenes de la carpeta imagenes_vit
# Estructura esperada:
#   imagenes_vit/clase_0/imagen1.png
#   imagenes_vit/clase_1/imagen2.png
# -----------------------------------------------------------------------------
class DatasetViT(Dataset):
    def __init__(self, carpeta, processor, transform=None):
        self.carpeta = Path(carpeta)
        self.processor = processor                          # Preprocesador del HF
        # Listar todas las clases (cada subcarpeta = una clase)
        self.clases = sorted([d.name for d in self.carpeta.iterdir() if d.is_dir()])
        self.clase_a_idx = {c: i for i, c in enumerate(self.clases)}
        # Recopilar todas las rutas de imagen junto con su etiqueta
        self.muestras = []
        for clase in self.clases:
            for img_path in (self.carpeta / clase).glob("*"):
                if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                    self.muestras.append((img_path, self.clase_a_idx[clase]))
        self.transform = transform

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, idx):
        img_path, label = self.muestras[idx]
        img = Image.open(img_path).convert("RGB")           # Forzar 3 canales
        # El processor de ViT normaliza con la media/desv del modelo pre-entrenado
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), torch.tensor(label)

# -----------------------------------------------------------------------------
# Construcción del modelo ViT pre-entrenado y reemplazo de la cabeza
# -----------------------------------------------------------------------------
def construir_vit(num_clases=2):
    # Cargar processor (encarga el resize a 224 y la normalización ImageNet)
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
    # Cargar el modelo y sustituir la cabeza de clasificación por num_clases
    modelo = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224",
        num_labels=num_clases,
        ignore_mismatched_sizes=True,                       # Permite cambiar el head
    )
    return modelo.to(DEVICE), processor

# -----------------------------------------------------------------------------
# Bucle de entrenamiento estándar
# -----------------------------------------------------------------------------
def entrenar_vit(modelo, loader, epochs=5, lr=2e-5):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr)
    criterio = nn.CrossEntropyLoss()
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for pixels, etiquetas in loader:
            pixels = pixels.to(DEVICE)
            etiquetas = etiquetas.to(DEVICE)
            optimizador.zero_grad()
            # ViT devuelve logits en .logits
            salida = modelo(pixel_values=pixels).logits
            perdida = criterio(salida, etiquetas)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        print(f"[ViT] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# -----------------------------------------------------------------------------
# Llamadas (NO ejecutar todavía, solo cuando coloques imágenes en la carpeta)
# -----------------------------------------------------------------------------
#modelo_vit, processor_vit = construir_vit(num_clases=2)
#dataset_vit = DatasetViT("/content/transformers_medicos/imagenes_vit", processor_vit)
#loader_vit = DataLoader(dataset_vit, batch_size=8, shuffle=True, num_workers=2)
#modelo_vit = entrenar_vit(modelo_vit, loader_vit, epochs=5)
#torch.save(modelo_vit.state_dict(), "/content/vit_finetuned.pth")
# Solo entrenar si la carpeta tiene imágenes (evita error si no se descargaron)
_carpeta_vit = Path("/content/transformers_medicos/imagenes_vit")
_tiene_imgs = any(p.is_file() and p.suffix.lower() in [".png",".jpg",".jpeg"]
                  for p in _carpeta_vit.rglob("*"))
if _tiene_imgs:
    modelo_vit, processor_vit = construir_vit(num_clases=2)
    dataset_vit = DatasetViT(str(_carpeta_vit), processor_vit)
    # num_workers=0 evita problemas de pickle en Colab
    loader_vit = DataLoader(dataset_vit, batch_size=8, shuffle=True, num_workers=0)
    modelo_vit = entrenar_vit(modelo_vit, loader_vit, epochs=2)   # 2 épocas demo
    torch.save(modelo_vit.state_dict(), "/content/vit_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_vit vacía: ejecuta primero la celda de descarga.")
    modelo_vit, processor_vit, loader_vit = None, None, None

In [ ]:
# =============================================================================
# CELDA 3: SWIN TRANSFORMER
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_swin
# Tipo de imagen:       Histopatología 2D (.png/.tiff), parches 224x224 RGB
# Tarea:                Clasificación tumor benigno/maligno
# Modelo base:          microsoft/swin-base-patch4-window7-224
# =============================================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoImageProcessor, SwinForImageClassification
from pathlib import Path

# -----------------------------------------------------------------------------
# Dataset para imágenes histopatológicas
# El Swin usa ventanas desplazadas; conviene resize a 224x224 exactos
# -----------------------------------------------------------------------------
class DatasetSwin(Dataset):
    def __init__(self, carpeta, processor):
        self.carpeta = Path(carpeta)
        self.processor = processor
        self.clases = sorted([d.name for d in self.carpeta.iterdir() if d.is_dir()])
        self.clase_a_idx = {c: i for i, c in enumerate(self.clases)}
        self.muestras = []
        for clase in self.clases:
            for img_path in (self.carpeta / clase).glob("*"):
                if img_path.suffix.lower() in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
                    self.muestras.append((img_path, self.clase_a_idx[clase]))

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, idx):
        img_path, label = self.muestras[idx]
        img = Image.open(img_path).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), torch.tensor(label)

# -----------------------------------------------------------------------------
# Construcción del Swin Transformer base pre-entrenado en ImageNet-1k
# -----------------------------------------------------------------------------
def construir_swin(num_clases=2):
    nombre_modelo = "microsoft/swin-base-patch4-window7-224"
    processor = AutoImageProcessor.from_pretrained(nombre_modelo)
    modelo = SwinForImageClassification.from_pretrained(
        nombre_modelo,
        num_labels=num_clases,
        ignore_mismatched_sizes=True,
    )
    return modelo.to(DEVICE), processor

# -----------------------------------------------------------------------------
# Entrenamiento (mismo esquema que ViT pero con menor LR para Swin base)
# -----------------------------------------------------------------------------
def entrenar_swin(modelo, loader, epochs=5, lr=1e-5):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr)
    # Scheduler con calentamiento simple (mejora estabilidad en Swin)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=epochs)
    criterio = nn.CrossEntropyLoss()
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for pixels, etiquetas in loader:
            pixels = pixels.to(DEVICE)
            etiquetas = etiquetas.to(DEVICE)
            optimizador.zero_grad()
            logits = modelo(pixel_values=pixels).logits
            perdida = criterio(logits, etiquetas)
            perdida.backward()
            # Recorte de gradiente para Swin (evita explosión con LR alto)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            optimizador.step()
            perdida_total += perdida.item()
        scheduler.step()
        print(f"[Swin] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# Ejemplo de uso (no ejecutar hasta tener imágenes):
#modelo_swin, proc_swin = construir_swin(num_clases=2)
#dset_swin = DatasetSwin("/content/transformers_medicos/imagenes_swin", proc_swin)
#loader_swin = DataLoader(dset_swin, batch_size=8, shuffle=True)
#entrenar_swin(modelo_swin, loader_swin, epochs=5)
_carpeta_swin = Path("/content/transformers_medicos/imagenes_swin")
_tiene = any(p.is_file() and p.suffix.lower() in [".png",".jpg",".tif",".tiff"]
             for p in _carpeta_swin.rglob("*"))
if _tiene:
    modelo_swin, proc_swin = construir_swin(num_clases=2)
    dset_swin = DatasetSwin(str(_carpeta_swin), proc_swin)
    loader_swin = DataLoader(dset_swin, batch_size=8, shuffle=True, num_workers=0)
    modelo_swin = entrenar_swin(modelo_swin, loader_swin, epochs=2)
    torch.save(modelo_swin.state_dict(), "/content/swin_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_swin vacía.")
    modelo_swin, proc_swin, loader_swin = None, None, None

In [ ]:
# =============================================================================
# CELDA 4: BEiT v2 (Bidirectional Encoder representation from Image Transformers)
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_beit
# Tipo de imagen:       Dermatoscopía 2D (.jpg), 224x224 RGB
# Tarea:                Clasificación melanoma vs nevo / multiclase ISIC
# Modelo base:          microsoft/beit-base-patch16-224 (BEiT v1, compatible)
#                       Para v2 puro: usar checkpoint custom de timm
# =============================================================================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import BeitImageProcessor, BeitForImageClassification
import timm                                                  # Para BEiT v2 oficial
from pathlib import Path

# -----------------------------------------------------------------------------
# Dataset para imágenes dermatoscópicas
# -----------------------------------------------------------------------------
class DatasetBEiT(Dataset):
    def __init__(self, carpeta, processor):
        self.carpeta = Path(carpeta)
        self.processor = processor
        self.clases = sorted([d.name for d in self.carpeta.iterdir() if d.is_dir()])
        self.clase_a_idx = {c: i for i, c in enumerate(self.clases)}
        self.muestras = []
        for clase in self.clases:
            for img_path in (self.carpeta / clase).glob("*"):
                if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                    self.muestras.append((img_path, self.clase_a_idx[clase]))

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, idx):
        img_path, label = self.muestras[idx]
        img = Image.open(img_path).convert("RGB")
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), torch.tensor(label)

# -----------------------------------------------------------------------------
# OPCIÓN A: BEiT vía Hugging Face (más simple, BEiT v1 oficial)
# -----------------------------------------------------------------------------
def construir_beit_hf(num_clases=2):
    nombre = "microsoft/beit-base-patch16-224"
    processor = BeitImageProcessor.from_pretrained(nombre)
    modelo = BeitForImageClassification.from_pretrained(
        nombre,
        num_labels=num_clases,
        ignore_mismatched_sizes=True,
    )
    return modelo.to(DEVICE), processor

# -----------------------------------------------------------------------------
# OPCIÓN B: BEiT v2 vía timm (versión real v2 con tokenizer VQ-KD)
# Requiere: pip install timm
# -----------------------------------------------------------------------------
def construir_beit_v2_timm(num_clases=2):
    # 'beitv2_base_patch16_224' es el checkpoint oficial v2 en timm
    modelo = timm.create_model(
        "beitv2_base_patch16_224.in1k_ft_in22k_in1k",
        pretrained=True,
        num_classes=num_clases,                              # Reemplaza head
    )
    # Construir un processor manual equivalente (timm no trae processor HF)
    from torchvision import transforms
    processor = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5],
                             std=[0.5, 0.5, 0.5]),           # Normalización BEiT
    ])
    return modelo.to(DEVICE), processor

# -----------------------------------------------------------------------------
# Entrenamiento BEiT
# -----------------------------------------------------------------------------
def entrenar_beit(modelo, loader, epochs=5, lr=2e-5, modo="hf"):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=0.05)
    criterio = nn.CrossEntropyLoss(label_smoothing=0.1)      # Suavizado típico BEiT
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for pixels, etiquetas in loader:
            pixels = pixels.to(DEVICE)
            etiquetas = etiquetas.to(DEVICE)
            optimizador.zero_grad()
            # Diferencia: HF devuelve dataclass, timm devuelve tensor directo
            if modo == "hf":
                logits = modelo(pixel_values=pixels).logits
            else:
                logits = modelo(pixels)
            perdida = criterio(logits, etiquetas)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        print(f"[BEiT] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# Ejemplo (descomentar cuando haya imágenes):
#modelo_beit, proc_beit = construir_beit_hf(num_clases=2)
#dset = DatasetBEiT("/content/transformers_medicos/imagenes_beit", proc_beit)
#loader = DataLoader(dset, batch_size=8, shuffle=True)
#entrenar_beit(modelo_beit, loader, epochs=5, modo="hf")
_carpeta_beit = Path("/content/transformers_medicos/imagenes_beit")
_tiene = any(p.is_file() and p.suffix.lower() in [".png",".jpg",".jpeg"]
             for p in _carpeta_beit.rglob("*"))
if _tiene:
    modelo_beit, proc_beit = construir_beit_hf(num_clases=2)
    dset_beit = DatasetBEiT(str(_carpeta_beit), proc_beit)
    loader_beit = DataLoader(dset_beit, batch_size=8, shuffle=True, num_workers=0)
    modelo_beit = entrenar_beit(modelo_beit, loader_beit, epochs=2, modo="hf")
    torch.save(modelo_beit.state_dict(), "/content/beit_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_beit vacía.")
    modelo_beit, proc_beit, loader_beit = None, None, None

In [ ]:
# =============================================================================
# CELDA 5: SWIN UNETR (Segmentación volumétrica 3D)
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_swin_unetr
# Tipo de imagen:       Volúmenes NIfTI (.nii.gz). Ej: BraTS, BTCV
# Tarea:                Segmentación 3D multi-clase (tumores, órganos)
# Modelo:               MONAI - SwinUNETR
# =============================================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.data import Dataset, decollate_batch
#from monai.transforms import (
#    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
#    ScaleIntensityRanged, CropForegroundd, RandCropByPosNegLabeld,
#    RandFlipd, RandShiftIntensityd, ToTensord, EnsureTyped
#)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    NormalizeIntensityd, CropForegroundd, RandCropByPosNegLabeld,
    RandFlipd, RandShiftIntensityd, ToTensord, EnsureTyped, Lambdad
)
from pathlib import Path
import glob

# -----------------------------------------------------------------------------
# Construcción de la lista de pares (imagen, etiqueta) en formato MONAI
# Espera estructura:
#   imagenes_swin_unetr/imagenesTr/*.nii.gz
#   imagenes_swin_unetr/etiquetasTr/*.nii.gz
# -----------------------------------------------------------------------------
def cargar_lista_swin_unetr(carpeta_raiz):
    raiz = Path(carpeta_raiz)
    imagenes = sorted(glob.glob(str(raiz / "imagenesTr" / "*.nii.gz")))
    etiquetas = sorted(glob.glob(str(raiz / "etiquetasTr" / "*.nii.gz")))
    # Cada elemento de MONAI es un dict con claves 'image' y 'label'
    return [{"image": img, "label": lbl} for img, lbl in zip(imagenes, etiquetas)]

# -----------------------------------------------------------------------------
# Pipeline de transformaciones para 3D (resampling, normalización, recorte)
# -----------------------------------------------------------------------------
"""def transformaciones_swin_unetr(roi=(96, 96, 96)):
    return Compose([
        LoadImaged(keys=["image", "label"]),                 # Carga NIfTI
        EnsureChannelFirstd(keys=["image", "label"]),        # Añade dim de canal
        # Resamplea a espaciado isotrópico de 1.5mm (típico para BTCV)
        Spacingd(keys=["image", "label"],
                 pixdim=(1.5, 1.5, 2.0),
                 mode=("bilinear", "nearest")),
        Orientationd(keys=["image", "label"], axcodes="RAS"),# Orientación estándar
        # Normaliza intensidades en rango HU típico para TC abdominal
        ScaleIntensityRanged(keys=["image"], a_min=-175, a_max=250,
                             b_min=0.0, b_max=1.0, clip=True),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        # Crop aleatorio balanceado pos/neg (mejora aprendizaje)
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label",
            spatial_size=roi, pos=1, neg=1, num_samples=2,
            image_key="image", image_threshold=0,
        ),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.2),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        EnsureTyped(keys=["image", "label"]),
        ToTensord(keys=["image", "label"]),
    ])"""
def transformaciones_swin_unetr(roi=(96, 96, 96)):
    """
    Pipeline para BraTS (RM cerebral). Cambios respecto a TC:
      - NormalizeIntensityd en lugar de ScaleIntensityRanged (no hay HU en RM)
      - Spacing isotrópico 1mm (BraTS ya viene re-muestreado)
      - Lambdad remapea la etiqueta 4 -> 3 (BraTS usa 0,1,2,4 -> consolidamos a 0..3)
    """
    return Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"],
                 pixdim=(1.0, 1.0, 1.0),
                 mode=("bilinear", "nearest")),
        # Remapeo BraTS: 0=fondo, 1=NCR/NET, 2=ED, 4=ET -> 3
        Lambdad(keys="label", func=lambda x: (x == 4) * 3 + (x == 2) * 2 + (x == 1) * 1),
        NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        RandCropByPosNegLabeld(
            keys=["image", "label"], label_key="label",
            spatial_size=roi, pos=1, neg=1, num_samples=1,
            image_key="image", image_threshold=0,
        ),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.2),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        EnsureTyped(keys=["image", "label"]),
        ToTensord(keys=["image", "label"]),
    ])
# -----------------------------------------------------------------------------
# Construcción del modelo Swin UNETR
# -----------------------------------------------------------------------------
def construir_swin_unetr(num_clases=14, roi=(96, 96, 96)):
    modelo = SwinUNETR(
        img_size=roi,
        in_channels=1,                                       # 1 canal (TC) o 4 (BraTS)
        out_channels=num_clases,                             # Nº de etiquetas + fondo
        feature_size=48,                                     # 48 = base, 96 = grande
        use_checkpoint=True,                                 # Ahorra VRAM (T4 16GB)
    )
    return modelo.to(DEVICE)

# -----------------------------------------------------------------------------
# Bucle de entrenamiento con DiceCE Loss (combina Dice + CrossEntropy)
# -----------------------------------------------------------------------------
def entrenar_swin_unetr(modelo, loader, epochs=50, lr=1e-4):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-5)
    # to_onehot_y convierte etiquetas a one-hot; softmax en salida del modelo
    criterio = DiceCELoss(to_onehot_y=True, softmax=True)
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for batch in loader:
            imgs = batch["image"].to(DEVICE)
            etis = batch["label"].to(DEVICE)
            optimizador.zero_grad()
            salida = modelo(imgs)                            # [B, C, D, H, W]
            perdida = criterio(salida, etis)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        print(f"[SwinUNETR] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# Solo entrena si la carpeta tiene volúmenes NIfTI; si no, evita el error
_datos = cargar_lista_swin_unetr("/content/transformers_medicos/imagenes_swin_unetr")
if len(_datos) > 0:
    tr = transformaciones_swin_unetr()
    ds_su = Dataset(data=_datos, transform=tr)
    # batch_size=1 y num_workers=0 para evitar OOM en T4 (16 GB)
    loader_su = DataLoader(ds_su, batch_size=1, shuffle=True, num_workers=0)
    # num_clases=4 porque BraTS tras el remapeo tiene fondo + 3 sub-regiones
    modelo_su = construir_swin_unetr(num_clases=4)
    modelo_su = entrenar_swin_unetr(modelo_su, loader_su, epochs=2)  # demo
    torch.save(modelo_su.state_dict(), "/content/swin_unetr_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_swin_unetr vacía: requiere Kaggle (BraTS).")
    modelo_su, loader_su = None, None

In [ ]:
# =============================================================================
# CELDA 6: TransUNet (Transformer + U-Net híbrido)
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_transunet
# Tipo de imagen:       Cortes 2D de TC/RM (.png) 224x224 escala de grises o RGB
# Tarea:                Segmentación 2D multi-clase (Synapse, ACDC)
# Implementación:       Custom (CNN encoder ResNet50 + ViT bottleneck + decoder U-Net)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tvm
from PIL import Image
import numpy as np
from pathlib import Path

# -----------------------------------------------------------------------------
# Dataset para cortes 2D con su máscara de segmentación
# Estructura:
#   imagenes_transunet/imagenes/*.png
#   imagenes_transunet/mascaras/*.png   (mismo nombre, valores enteros = clases)
# -----------------------------------------------------------------------------
class DatasetTransUNet(Dataset):
    def __init__(self, carpeta, tam=224):
        self.carpeta = Path(carpeta)
        self.imgs = sorted(list((self.carpeta / "imagenes").glob("*.png")))
        self.tam = tam

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        ruta_img = self.imgs[idx]
        ruta_msk = self.carpeta / "mascaras" / ruta_img.name
        # Carga imagen y máscara, redimensiona a (tam, tam)
        img = Image.open(ruta_img).convert("RGB").resize((self.tam, self.tam))
        msk = Image.open(ruta_msk).resize((self.tam, self.tam), Image.NEAREST)
        img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        msk = torch.from_numpy(np.array(msk)).long()         # Etiquetas enteras
        return img, msk

# -----------------------------------------------------------------------------
# Bloque Transformer estándar (atención multi-cabeza + MLP + LN residual)
# -----------------------------------------------------------------------------
class BloqueTransformer(nn.Module):
    def __init__(self, dim=768, heads=12, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(drop),
        )

    def forward(self, x):
        # Atención auto-residual
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, need_weights=False)
        x = x + h
        # MLP residual
        x = x + self.mlp(self.norm2(x))
        return x

# -----------------------------------------------------------------------------
# Modelo TransUNet completo: encoder ResNet50 -> ViT bottleneck -> decoder U-Net
# -----------------------------------------------------------------------------
class TransUNet(nn.Module):
    def __init__(self, num_clases=9, dim=768, n_bloques=12):
        super().__init__()
        # Encoder CNN (ResNet50 hasta layer4 da feature map de 14x14 con 2048 canales)
        resnet = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        self.encoder0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64
        self.pool0 = resnet.maxpool
        self.encoder1 = resnet.layer1                        # 256 canales
        self.encoder2 = resnet.layer2                        # 512
        self.encoder3 = resnet.layer3                        # 1024
        self.encoder4 = resnet.layer4                        # 2048, 14x14

        # Proyección a embedding del Transformer
        self.proj = nn.Conv2d(2048, dim, kernel_size=1)
        # Embeddings posicionales aprendibles para 14*14=196 tokens
        self.pos_embed = nn.Parameter(torch.zeros(1, 196, dim))
        # Pila de bloques Transformer
        self.bloques = nn.ModuleList([BloqueTransformer(dim) for _ in range(n_bloques)])
        self.norm_final = nn.LayerNorm(dim)

        # Decoder U-Net con skip connections desde el encoder
        self.dec4 = self._bloque_dec(dim,  512)              # 14->28
        self.dec3 = self._bloque_dec(512 + 1024, 256)        # 28->56
        self.dec2 = self._bloque_dec(256 + 512,  128)        # 56->112
        self.dec1 = self._bloque_dec(128 + 256,   64)        # 112->224
        self.salida = nn.Conv2d(64, num_clases, 1)           # Mapa final por clase

    def _bloque_dec(self, ch_in, ch_out):
        # Convolución + BN + ReLU + upsample x2
        return nn.Sequential(
            nn.Conv2d(ch_in, ch_out, 3, padding=1),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_out, ch_out, 3, padding=1),
            nn.BatchNorm2d(ch_out),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
        )

    def forward(self, x):
        # Encoder
        e0 = self.encoder0(x)                                # [B, 64, 112, 112]
        e0p = self.pool0(e0)                                 # [B, 64, 56, 56]
        e1 = self.encoder1(e0p)                              # [B, 256, 56, 56]
        e2 = self.encoder2(e1)                               # [B, 512, 28, 28]
        e3 = self.encoder3(e2)                               # [B, 1024, 14, 14]
        e4 = self.encoder4(e3)                               # [B, 2048, 7, 7]

        # Bottleneck Transformer (re-tokenización en 14x14)
        e4_up = F.interpolate(e4, size=(14, 14), mode="bilinear", align_corners=False)
        tokens = self.proj(e4_up).flatten(2).transpose(1, 2) # [B, 196, dim]
        tokens = tokens + self.pos_embed
        for bloque in self.bloques:
            tokens = bloque(tokens)
        tokens = self.norm_final(tokens)
        # Reconstruir mapa 2D 14x14
        B, N, C = tokens.shape
        feat = tokens.transpose(1, 2).reshape(B, C, 14, 14)

        # Decoder con skips (concatenación)
        d4 = self.dec4(feat)                                 # [B, 512, 28, 28]
        d3 = self.dec3(torch.cat([d4, e3], dim=1))           # [B, 256, 56, 56]
        # Ajuste de tamaño antes de concatenar e2 (28x28 vs 56x56)
        d3 = F.interpolate(d3, size=e2.shape[2:], mode="bilinear", align_corners=False)
        d2 = self.dec2(torch.cat([d3, e2], dim=1))           # [B, 128, 56->112]
        d2 = F.interpolate(d2, size=e1.shape[2:], mode="bilinear", align_corners=False)
        d1 = self.dec1(torch.cat([d2, e1], dim=1))           # [B, 64, 112->224]
        return self.salida(d1)                               # Logits [B, K, 224, 224]

# -----------------------------------------------------------------------------
# Entrenamiento con CrossEntropy + Dice (suma ponderada)
# -----------------------------------------------------------------------------
def dice_loss(logits, target, smooth=1.0):
    # Pérdida Dice multiclase mediante softmax + one-hot
    probs = F.softmax(logits, dim=1)
    n_clases = probs.shape[1]
    target_oh = F.one_hot(target, n_clases).permute(0, 3, 1, 2).float()
    inter = (probs * target_oh).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + target_oh.sum(dim=(2, 3))
    return 1 - ((2 * inter + smooth) / (union + smooth)).mean()

def entrenar_transunet(modelo, loader, epochs=30, lr=1e-4):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for img, msk in loader:
            img, msk = img.to(DEVICE), msk.to(DEVICE)
            optimizador.zero_grad()
            logits = modelo(img)
            # Pérdida combinada (factor 0.5/0.5 estándar)
            perdida = 0.5 * ce(logits, msk) + 0.5 * dice_loss(logits, msk)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        print(f"[TransUNet] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# Uso:
#modelo_tu = TransUNet(num_clases=9).to(DEVICE)
#ds = DatasetTransUNet("/content/transformers_medicos/imagenes_transunet")
#loader = DataLoader(ds, batch_size=4, shuffle=True)
#entrenar_transunet(modelo_tu, loader, epochs=30)
_carpeta_tu = Path("/content/transformers_medicos/imagenes_transunet/imagenes")
_tiene = _carpeta_tu.exists() and any(_carpeta_tu.glob("*.png"))
if _tiene:
    modelo_tu = TransUNet(num_clases=9).to(DEVICE)
    ds_tu = DatasetTransUNet("/content/transformers_medicos/imagenes_transunet")
    loader_tu = DataLoader(ds_tu, batch_size=4, shuffle=True, num_workers=0)
    modelo_tu = entrenar_transunet(modelo_tu, loader_tu, epochs=2)  # demo
    torch.save(modelo_tu.state_dict(), "/content/transunet_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_transunet vacía.")
    modelo_tu, loader_tu = None, None

In [ ]:
# =============================================================================
# CELDA 7: SWIN-UNET (Encoder-Decoder enteramente Transformer)
# Carpeta de imágenes:  /content/transformers_medicos/imagenes_swin_unet
# Tipo de imagen:       RM cardíaca 2D (.png) 224x224 escala de grises
# Tarea:                Segmentación 2D (ventrículos/miocardio - ACDC)
# Backbone:             Swin Transformer simétrico con Patch Expanding
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange
import timm                                                  # Para usar swin pre-entrenado
from pathlib import Path
from PIL import Image
import numpy as np

# -----------------------------------------------------------------------------
# Dataset (mismo formato que TransUNet)
# -----------------------------------------------------------------------------
class DatasetSwinUnet(Dataset):
    def __init__(self, carpeta, tam=224):
        self.carpeta = Path(carpeta)
        self.imgs = sorted(list((self.carpeta / "imagenes").glob("*.png")))
        self.tam = tam

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        ruta_img = self.imgs[idx]
        ruta_msk = self.carpeta / "mascaras" / ruta_img.name
        img = Image.open(ruta_img).convert("L").resize((self.tam, self.tam))
        msk = Image.open(ruta_msk).resize((self.tam, self.tam), Image.NEAREST)
        # Convertir a tensor [1, H, W] y replicar a 3 canales para usar Swin pre-entrenado
        img = torch.from_numpy(np.array(img)).unsqueeze(0).float() / 255.0
        img = img.repeat(3, 1, 1)
        msk = torch.from_numpy(np.array(msk)).long()
        return img, msk

# -----------------------------------------------------------------------------
# Bloque "Patch Expanding": opuesto al Patch Merging (upsample + reducción canales)
# -----------------------------------------------------------------------------
class PatchExpand(nn.Module):
    def __init__(self, dim, factor=2):
        super().__init__()
        self.expand = nn.Linear(dim, factor * dim, bias=False)
        self.norm = nn.LayerNorm(dim // factor)
        self.factor = factor

    def forward(self, x, H, W):
        # x: [B, H*W, C]
        B, L, C = x.shape
        x = self.expand(x)                                   # [B, L, 2C]
        x = x.view(B, H, W, 2 * C)
        # Reorganizar canales para duplicar resolución espacial
        x = rearrange(x, "b h w (p1 p2 c) -> b (h p1) (w p2) c", p1=2, p2=2)
        x = x.view(B, -1, C // 2)
        x = self.norm(x)
        return x, H * 2, W * 2

# -----------------------------------------------------------------------------
# Swin-Unet simplificado: usamos un Swin de timm como encoder y un decoder simétrico
# Para una implementación 100% fiel al paper original, ver:
# https://github.com/HuCaoFighting/Swin-Unet
# -----------------------------------------------------------------------------
class SwinUnet(nn.Module):
    def __init__(self, num_clases=4, img_size=224):
        super().__init__()
        # Encoder: Swin-Tiny pre-entrenado (4 etapas, dims [96, 192, 384, 768])
        self.encoder = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            features_only=True,                              # Devuelve mapas multi-escala
        )
        # Las dimensiones de salida de cada etapa
        dims = [96, 192, 384, 768]

        # Decoder: 3 niveles de Patch Expand + skip connections
        self.expand3 = nn.ConvTranspose2d(dims[3], dims[2], 2, stride=2)
        self.expand2 = nn.ConvTranspose2d(dims[2] * 2, dims[1], 2, stride=2)
        self.expand1 = nn.ConvTranspose2d(dims[1] * 2, dims[0], 2, stride=2)
        self.expand0 = nn.ConvTranspose2d(dims[0] * 2, dims[0], 4, stride=4)

        # Cabezal de salida con num_clases canales
        self.salida = nn.Conv2d(dims[0], num_clases, 1)

    def forward(self, x):
        # Encoder produce [B, H, W, C] por etapa; convertir a [B, C, H, W]
        feats = self.encoder(x)
        # timm devuelve listas en formato NHWC, las pasamos a NCHW
        feats = [f.permute(0, 3, 1, 2).contiguous() if f.shape[-1] in [96,192,384,768] else f
                 for f in feats]
        f1, f2, f3, f4 = feats                               # 56,28,14,7

        # Decoder con skips concatenadas
        d3 = self.expand3(f4)                                # 7 -> 14
        d3 = torch.cat([d3, f3], dim=1)
        d2 = self.expand2(d3)                                # 14 -> 28
        d2 = torch.cat([d2, f2], dim=1)
        d1 = self.expand1(d2)                                # 28 -> 56
        d1 = torch.cat([d1, f1], dim=1)
        d0 = self.expand0(d1)                                # 56 -> 224
        return self.salida(d0)                               # Logits

# -----------------------------------------------------------------------------
# Entrenamiento (mismo loss combinado CE + Dice que TransUNet)
# -----------------------------------------------------------------------------
def entrenar_swin_unet(modelo, loader, epochs=30, lr=5e-5):
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=1e-4)
    ce = nn.CrossEntropyLoss()
    modelo.train()
    for epoca in range(epochs):
        perdida_total = 0.0
        for img, msk in loader:
            img, msk = img.to(DEVICE), msk.to(DEVICE)
            optimizador.zero_grad()
            logits = modelo(img)
            perdida = 0.5 * ce(logits, msk) + 0.5 * dice_loss(logits, msk)
            perdida.backward()
            optimizador.step()
            perdida_total += perdida.item()
        print(f"[SwinUnet] Época {epoca+1}/{epochs} - Pérdida: {perdida_total/len(loader):.4f}")
    return modelo

# Uso:
# modelo_su = SwinUnet(num_clases=4).to(DEVICE)
# ds = DatasetSwinUnet("/content/transformers_medicos/imagenes_swin_unet")
# loader = DataLoader(ds, batch_size=8, shuffle=True)
# entrenar_swin_unet(modelo_su, loader, epochs=30)
_carpeta_swu = Path("/content/transformers_medicos/imagenes_swin_unet/imagenes")
_tiene = _carpeta_swu.exists() and any(_carpeta_swu.glob("*.png"))
if _tiene:
    # Renombrado a modelo_swu para no chocar con modelo_su (Swin UNETR)
    modelo_swu = SwinUnet(num_clases=4).to(DEVICE)
    ds_swu = DatasetSwinUnet("/content/transformers_medicos/imagenes_swin_unet")
    loader_swu = DataLoader(ds_swu, batch_size=8, shuffle=True, num_workers=0)
    modelo_swu = entrenar_swin_unet(modelo_swu, loader_swu, epochs=2)
    torch.save(modelo_swu.state_dict(), "/content/swin_unet_finetuned.pth")
else:
    print("⚠️ Carpeta imagenes_swin_unet vacía: requiere Kaggle (ACDC).")
    modelo_swu, loader_swu = None, None

In [ ]:
# =============================================================================
# CELDA 8: GENERACIÓN DE VALORES SHAP PARA LOS MODELOS DE CLASIFICACIÓN
# Funciona con ViT, Swin, BEiT (entradas 2D).
# Para Swin UNETR / TransUNet / Swin-Unet (segmentación) se aplica SHAP por píxel.
# =============================================================================
import shap
import numpy as np
import torch
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# Wrapper genérico: convierte un modelo PyTorch en una función f(x_numpy) -> probs
# Es lo que SHAP necesita como predict function
# -----------------------------------------------------------------------------
def envolver_modelo_clasif(modelo, modo="hf"):
    modelo.eval()
    def f(x_np):
        # x_np: [N, H, W, 3] uint8 o float -> tensor [N, 3, H, W]
        x = torch.from_numpy(x_np).float()
        if x.shape[-1] == 3:
            x = x.permute(0, 3, 1, 2)
        x = x.to(DEVICE)
        with torch.no_grad():
            if modo == "hf":
                logits = modelo(pixel_values=x).logits
            else:
                logits = modelo(x)
            probs = F.softmax(logits, dim=1).cpu().numpy()
        return probs
    return f

# -----------------------------------------------------------------------------
# Explicador SHAP basado en Partition (recomendado para imágenes)
# Usa máscaras inpainting para perturbar regiones de la imagen
# -----------------------------------------------------------------------------
def explicar_shap(modelo, imagenes_np, etiquetas_clase, modo="hf", max_evals=2000):
    """
    imagenes_np : array [N, H, W, 3] con imágenes de prueba
    etiquetas_clase : nombres de clases (lista de str)
    """
    f = envolver_modelo_clasif(modelo, modo=modo)
    # Masker basado en inpainting telea (sustituye píxeles según vecindario)
    masker = shap.maskers.Image("inpaint_telea", imagenes_np[0].shape)
    explainer = shap.Explainer(f, masker, output_names=etiquetas_clase)
    # Genera valores SHAP por píxel y por clase
    shap_values = explainer(imagenes_np, max_evals=max_evals,
                            batch_size=8, outputs=shap.Explanation.argsort.flip[:1])
    return shap_values

# -----------------------------------------------------------------------------
# Visualización de SHAP (overlay sobre la imagen original)
# -----------------------------------------------------------------------------
def visualizar_shap(shap_values, imagenes_np):
    shap.image_plot(shap_values, imagenes_np)

# Ejemplo de uso (después de entrenar un clasificador):
# imgs = np.stack([np.array(Image.open(p).resize((224,224))) for p in lista_paths])
# sv = explicar_shap(modelo_vit, imgs, etiquetas_clase=["sano","neumonia"])
# visualizar_shap(sv, imgs)

In [ ]:
# =============================================================================
# CELDA 9: MÉTRICAS DE EVALUACIÓN DE EXPLICACIONES
# Cubre: Fidelidad, Robustez, Localización, Complejidad
# =============================================================================
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import jaccard_score

# =============================================================================
# 9.1  FIDELIDAD
# =============================================================================

# -----------------------------------------------------------------------------
# Faithfulness Correlation: correlación entre la suma de atribuciones de las
# regiones perturbadas y la caída en la probabilidad del modelo.
# -----------------------------------------------------------------------------
def faithfulness_correlation(modelo, imagen, atribuciones, n_subset=30, tam_subset=50):
    """
    imagen: tensor [1, C, H, W] en GPU
    atribuciones: array [H, W] con valores SHAP del píxel
    """
    modelo.eval()
    H, W = atribuciones.shape
    px_total = H * W
    prob_orig = F.softmax(modelo(imagen).logits, dim=1).max().item()
    sumas_attr, deltas_pred = [], []
    for _ in range(n_subset):
        # Selecciona índices aleatorios para perturbar
        idx = np.random.choice(px_total, tam_subset, replace=False)
        ys, xs = np.unravel_index(idx, (H, W))
        img_pert = imagen.clone()
        # Reemplaza por la media de la imagen (perturbación baseline)
        img_pert[..., ys, xs] = imagen.mean()
        with torch.no_grad():
            prob_pert = F.softmax(modelo(img_pert).logits, dim=1).max().item()
        sumas_attr.append(atribuciones[ys, xs].sum())
        deltas_pred.append(prob_orig - prob_pert)
    # Correlación de Pearson: si las atribuciones son fieles, correlación alta
    return pearsonr(sumas_attr, deltas_pred)[0]

# -----------------------------------------------------------------------------
# Insertion / Deletion AUC
# Insertion: añade píxeles más importantes progresivamente, mide cómo SUBE la prob
# Deletion : elimina píxeles más importantes, mide cómo BAJA la prob
# -----------------------------------------------------------------------------
def insertion_deletion_auc(modelo, imagen, atribuciones, pasos=50, modo="deletion"):
    modelo.eval()
    H, W = atribuciones.shape
    flat_attr = atribuciones.flatten()
    # Orden de píxeles según importancia descendente
    orden = np.argsort(-flat_attr) if modo == "deletion" else np.argsort(-flat_attr)
    img_base = imagen.clone()
    if modo == "insertion":
        # Empieza con imagen completamente borrosa/cero
        img_base = torch.zeros_like(imagen)
    px_por_paso = len(orden) // pasos
    probs = []
    for paso in range(pasos):
        idx = orden[:(paso + 1) * px_por_paso]
        ys, xs = np.unravel_index(idx, (H, W))
        if modo == "deletion":
            # Reemplaza los píxeles importantes por 0
            img_mod = imagen.clone()
            img_mod[..., ys, xs] = 0
        else:
            # En insertion, copia los píxeles importantes desde la original
            img_mod = img_base.clone()
            img_mod[..., ys, xs] = imagen[..., ys, xs]
        with torch.no_grad():
            p = F.softmax(modelo(img_mod).logits, dim=1).max().item()
        probs.append(p)
    # AUC ≈ media de probabilidades a lo largo de los pasos
    return np.trapz(probs) / pasos

# -----------------------------------------------------------------------------
# Pixel-Flipping (PF): variante de Deletion que voltea los píxeles más
# importantes y mide la curva de degradación de la predicción.
# -----------------------------------------------------------------------------
def pixel_flipping(modelo, imagen, atribuciones, pasos=20):
    modelo.eval()
    H, W = atribuciones.shape
    orden = np.argsort(-atribuciones.flatten())
    px_por_paso = len(orden) // pasos
    curva = []
    img = imagen.clone()
    for paso in range(pasos):
        idx = orden[paso * px_por_paso:(paso + 1) * px_por_paso]
        ys, xs = np.unravel_index(idx, (H, W))
        # "Flip" -> sustituye por el valor opuesto (1 - x si normalizado)
        img[..., ys, xs] = 1.0 - img[..., ys, xs]
        with torch.no_grad():
            p = F.softmax(modelo(img).logits, dim=1).max().item()
        curva.append(p)
    return curva                                             # Lista de prob por paso

# =============================================================================
# 9.2  ROBUSTEZ Y CONSISTENCIA
# =============================================================================

# -----------------------------------------------------------------------------
# Sensitivity-N: correlación entre suma de atribuciones de N píxeles y el
# cambio en la salida cuando se perturban esos N píxeles
# -----------------------------------------------------------------------------
def sensitivity_n(modelo, imagen, atribuciones, N=100, repeticiones=50):
    modelo.eval()
    H, W = atribuciones.shape
    sums, deltas = [], []
    prob_orig = F.softmax(modelo(imagen).logits, dim=1).max().item()
    for _ in range(repeticiones):
        idx = np.random.choice(H * W, N, replace=False)
        ys, xs = np.unravel_index(idx, (H, W))
        img_p = imagen.clone(); img_p[..., ys, xs] = 0
        with torch.no_grad():
            prob_p = F.softmax(modelo(img_p).logits, dim=1).max().item()
        sums.append(atribuciones[ys, xs].sum())
        deltas.append(prob_orig - prob_p)
    return pearsonr(sums, deltas)[0]

# -----------------------------------------------------------------------------
# Robustez ante perturbaciones: añade ruido gaussiano y mide cuánto cambian
# las atribuciones (distancia L2 normalizada)
# -----------------------------------------------------------------------------
def robustez_perturbacion(funcion_attr, imagen, sigma=0.05, intentos=10):
    """
    funcion_attr: callable(imagen) -> mapa de atribuciones [H, W]
    """
    base = funcion_attr(imagen)
    distancias = []
    for _ in range(intentos):
        ruido = torch.randn_like(imagen) * sigma
        attr_pert = funcion_attr(imagen + ruido)
        # Distancia L2 entre mapas, normalizada por la norma del original
        d = np.linalg.norm(attr_pert - base) / (np.linalg.norm(base) + 1e-8)
        distancias.append(d)
    return np.mean(distancias)                               # Cuanto menor, mejor

# -----------------------------------------------------------------------------
# Estabilidad: varianza de las atribuciones a través de varias ejecuciones
# (útil cuando SHAP usa muestreo estocástico)
# -----------------------------------------------------------------------------
def estabilidad_ejecuciones(funcion_attr, imagen, n_ejec=5):
    mapas = np.stack([funcion_attr(imagen) for _ in range(n_ejec)])
    # Varianza media píxel a píxel
    return mapas.var(axis=0).mean()

# =============================================================================
# 9.3  LOCALIZACIÓN CLÍNICA (vs máscara de radiólogo)
# =============================================================================

# -----------------------------------------------------------------------------
# IoU entre el mapa de atribución binarizado y la máscara experta
# -----------------------------------------------------------------------------
def iou_vs_radiologo(atribuciones, mascara_gt, percentil=80):
    # Binariza la atribución: solo el percentil 80 superior cuenta
    umbral = np.percentile(atribuciones, percentil)
    mapa_bin = (atribuciones >= umbral).astype(np.uint8)
    return jaccard_score(mascara_gt.flatten(), mapa_bin.flatten())

# -----------------------------------------------------------------------------
# Coeficiente Dice (segmentación / SHAP-binarizado)
# -----------------------------------------------------------------------------
def dice_score(pred, target, eps=1e-6):
    pred = pred.astype(bool); target = target.astype(bool)
    inter = np.logical_and(pred, target).sum()
    return (2 * inter + eps) / (pred.sum() + target.sum() + eps)

# -----------------------------------------------------------------------------
# Pointing Game: el píxel con mayor atribución debe caer dentro de la máscara GT
# -----------------------------------------------------------------------------
def pointing_game(atribuciones, mascara_gt):
    # Coordenada del máximo
    y, x = np.unravel_index(np.argmax(atribuciones), atribuciones.shape)
    return float(mascara_gt[y, x] > 0)                       # 1 si acierta, 0 si no

# =============================================================================
# 9.4  COMPLEJIDAD
# =============================================================================

# -----------------------------------------------------------------------------
# Sparsity / Complexity: entropía normalizada de las atribuciones
# Más baja = explicación más concentrada (mejor)
# -----------------------------------------------------------------------------
def sparsity_complexity(atribuciones, eps=1e-12):
    a = np.abs(atribuciones).flatten()
    p = a / (a.sum() + eps)
    entropia = -np.sum(p * np.log(p + eps))
    # Normaliza entre 0 y 1 (max = log(n))
    return entropia / np.log(len(p))

# -----------------------------------------------------------------------------
# Randomization Test: si aleatorizamos los pesos del modelo, las atribuciones
# deberían cambiar drásticamente (sino, la explicación es independiente del modelo)
# -----------------------------------------------------------------------------
def randomization_test(modelo, funcion_attr, imagen):
    import copy
    base = funcion_attr(imagen)
    # Copia profunda y aleatoriza pesos de la última capa
    modelo_random = copy.deepcopy(modelo)
    for p in modelo_random.parameters():
        with torch.no_grad():
            p.copy_(torch.randn_like(p))
    # Recalcula atribuciones con el modelo aleatorizado
    attr_rand = funcion_attr(imagen)                         # Asume cierre sobre modelo_random
    # Devuelve la correlación: bajo = bueno (las explicaciones cambian)
    return spearmanr(base.flatten(), attr_rand.flatten())[0]

In [ ]:
# =============================================================================
# CELDA 10: MÉTRICAS DE RENDIMIENTO CLÍNICO
# Clasificación: AUC, F1, sensibilidad, especificidad
# Segmentación: Dice, IoU, HD95
# =============================================================================
import numpy as np
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report
)
from medpy.metric.binary import hd95, dc, jc

# -----------------------------------------------------------------------------
# Métricas de clasificación
# y_true: enteros [N], y_pred_probs: probabilidades [N, C]
# -----------------------------------------------------------------------------
def metricas_clasificacion(y_true, y_pred_probs, umbral=0.5):
    y_pred = np.argmax(y_pred_probs, axis=1)
    # AUC: si binario, usar columna 1; si multi, "ovr"
    if y_pred_probs.shape[1] == 2:
        auc = roc_auc_score(y_true, y_pred_probs[:, 1])
    else:
        auc = roc_auc_score(y_true, y_pred_probs, multi_class="ovr")
    # Sensibilidad = recall_pos, Especificidad = recall_neg
    cm = confusion_matrix(y_true, y_pred)
    tn = cm[0, 0]; fp = cm[0, 1]; fn = cm[1, 0]; tp = cm[1, 1] if cm.shape[0]>1 else 0
    sensibilidad = tp / (tp + fn + 1e-8)
    especificidad = tn / (tn + fp + 1e-8)
    return {
        "AUC": auc,
        "F1": f1_score(y_true, y_pred, average="weighted"),
        "Sensibilidad": sensibilidad,
        "Especificidad": especificidad,
        "Reporte": classification_report(y_true, y_pred, zero_division=0),
    }

# -----------------------------------------------------------------------------
# Métricas de segmentación (binaria o multi-clase mediante promedio)
# pred, gt: arrays binarios [H, W] o [D, H, W]
# -----------------------------------------------------------------------------
def metricas_segmentacion(pred, gt):
    # Comprueba que existe al menos un voxel positivo (HD95 falla si vacío)
    if pred.sum() == 0 or gt.sum() == 0:
        return {"Dice": 0.0, "IoU": 0.0, "HD95": np.inf}
    return {
        "Dice": dc(pred, gt),                                # Coef. Dice
        "IoU":  jc(pred, gt),                                # Jaccard / IoU
        "HD95": hd95(pred, gt),                              # Hausdorff 95
    }

# -----------------------------------------------------------------------------
# Wrapper para evaluar segmentación multi-clase (promedia por clase)
# -----------------------------------------------------------------------------
def metricas_seg_multiclase(pred_mask, gt_mask, n_clases):
    """
    pred_mask, gt_mask: arrays con valores enteros 0..n_clases-1
    Devuelve dict con métricas por clase + media (excluye clase 0 = fondo)
    """
    resultados = {}
    dices, ious, hd95s = [], [], []
    for c in range(1, n_clases):
        pred_c = (pred_mask == c).astype(np.uint8)
        gt_c = (gt_mask == c).astype(np.uint8)
        m = metricas_segmentacion(pred_c, gt_c)
        resultados[f"clase_{c}"] = m
        dices.append(m["Dice"]); ious.append(m["IoU"])
        if not np.isinf(m["HD95"]): hd95s.append(m["HD95"])
    resultados["media"] = {
        "Dice": np.mean(dices),
        "IoU":  np.mean(ious),
        "HD95": np.mean(hd95s) if hd95s else np.inf,
    }
    return resultados

In [ ]:
# =============================================================================
# CELDA 11: PIPELINE QUE COMBINA TODO LO ANTERIOR
# Ejecuta: predicciones -> SHAP -> métricas SHAP -> métricas clínicas
# Para cada modelo entrenado en celdas previas
# =============================================================================
import pandas as pd
import torch
import numpy as np

def evaluar_modelo_completo(
        modelo,                  # Modelo PyTorch ya entrenado
        loader_test,             # DataLoader con imágenes de test
        nombre_modelo,           # str: "ViT", "Swin", etc.
        modo="hf",               # "hf" para HuggingFace, "timm" o "custom" en otro caso
        es_segmentacion=False,   # True para Swin UNETR, TransUNet, Swin-Unet
        n_clases=2,
        mascaras_radiologo=None  # Lista de máscaras GT alineadas con loader_test
):
    """
    Devuelve un DataFrame con todas las métricas calculadas.
    """
    # ---------- Predicciones ----------
    modelo.eval()
    todas_probs, todas_etiq, todas_imgs = [], [], []
    with torch.no_grad():
        for batch in loader_test:
            imgs, etis = batch[0].to(DEVICE), batch[1]
            if not es_segmentacion:
                if modo == "hf":
                    logits = modelo(pixel_values=imgs).logits
                else:
                    logits = modelo(imgs)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                todas_probs.append(probs)
                todas_etiq.append(etis.numpy())
            else:
                # Segmentación: guardar predicciones argmax
                logits = modelo(imgs)
                pred = logits.argmax(dim=1).cpu().numpy()
                todas_probs.append(pred)
                todas_etiq.append(etis.numpy())
            todas_imgs.append(imgs.cpu().numpy())

    todas_probs = np.concatenate(todas_probs, axis=0)
    todas_etiq = np.concatenate(todas_etiq, axis=0)
    todas_imgs = np.concatenate(todas_imgs, axis=0)

    resultados = {"modelo": nombre_modelo}

    # ---------- Métricas clínicas ----------
    if not es_segmentacion:
        m_clin = metricas_clasificacion(todas_etiq, todas_probs)
        resultados.update({k: v for k, v in m_clin.items() if k != "Reporte"})
    else:
        # Para segmentación, agregar Dice/IoU/HD95 por imagen y promediar
        dices, ious, hds = [], [], []
        for p, g in zip(todas_probs, todas_etiq):
            mres = metricas_seg_multiclase(p, g, n_clases)
            dices.append(mres["media"]["Dice"])
            ious.append(mres["media"]["IoU"])
            hds.append(mres["media"]["HD95"])
        resultados.update({
            "Dice": np.mean(dices),
            "IoU":  np.mean(ious),
            "HD95": np.mean([h for h in hds if not np.isinf(h)]) if hds else np.inf,
        })

    # ---------- SHAP (solo clasificación: las imágenes 2D son razonables) ----------
    if not es_segmentacion:
        # Tomar muestra reducida para SHAP (es costoso)
        muestra = todas_imgs[:8].transpose(0, 2, 3, 1)       # [N, H, W, C]
        sv = explicar_shap(modelo, muestra,
                           etiquetas_clase=[f"c{i}" for i in range(n_clases)],
                           modo=modo, max_evals=500)
        # Convertir a mapas de atribución 2D (norma de los canales)
        atribs = np.linalg.norm(sv.values[..., 0], axis=-1)  # [N, H, W]

        # Métricas SHAP promediadas sobre la muestra
        fc, idel, iins, pf, sn = [], [], [], [], []
        for i, attr in enumerate(atribs):
            img_t = torch.from_numpy(todas_imgs[i:i+1]).to(DEVICE)
            fc.append(faithfulness_correlation(modelo, img_t, attr))
            idel.append(insertion_deletion_auc(modelo, img_t, attr, modo="deletion"))
            iins.append(insertion_deletion_auc(modelo, img_t, attr, modo="insertion"))
            pf.append(np.mean(pixel_flipping(modelo, img_t, attr)))
            sn.append(sensitivity_n(modelo, img_t, attr))
        resultados["FaithCorr"]  = np.nanmean(fc)
        resultados["DeletionAUC"] = np.nanmean(idel)
        resultados["InsertionAUC"] = np.nanmean(iins)
        resultados["PixelFlipping"] = np.nanmean(pf)
        resultados["SensitivityN"] = np.nanmean(sn)
        resultados["Sparsity"]   = np.mean([sparsity_complexity(a) for a in atribs])

        # Localización si tenemos máscaras de radiólogo
        if mascaras_radiologo is not None:
            ious_xai, dices_xai, pgs = [], [], []
            for attr, gt in zip(atribs, mascaras_radiologo):
                ious_xai.append(iou_vs_radiologo(attr, gt))
                dices_xai.append(dice_score(attr > np.percentile(attr, 80), gt))
                pgs.append(pointing_game(attr, gt))
            resultados["IoU_radiologo"] = np.mean(ious_xai)
            resultados["Dice_XAI"] = np.mean(dices_xai)
            resultados["PointingGame"] = np.mean(pgs)

    return pd.DataFrame([resultados])



In [ ]:
# =============================================================================
# CELDA FINAL: ORQUESTACIÓN DE EVALUACIÓN
# Solo evalúa los modelos que se entrenaron (los que tienen variables != None)
# =============================================================================
import pandas as pd

dfs = []

# --- Clasificadores: reusan el mismo loader que se usó para entrenar (demo) ---
if modelo_vit is not None and loader_vit is not None:
    dfs.append(evaluar_modelo_completo(modelo_vit, loader_vit, "ViT", modo="hf"))

if modelo_swin is not None and loader_swin is not None:
    dfs.append(evaluar_modelo_completo(modelo_swin, loader_swin, "Swin", modo="hf"))

if modelo_beit is not None and loader_beit is not None:
    dfs.append(evaluar_modelo_completo(modelo_beit, loader_beit, "BEiT", modo="hf"))

# --- Segmentadores ---
if modelo_su is not None and loader_su is not None:
    # n_clases=4 (BraTS tras remap), modo="custom" porque MONAI no es HF
    dfs.append(evaluar_modelo_completo(
        modelo_su, loader_su, "SwinUNETR",
        modo="custom", es_segmentacion=True, n_clases=4))

if modelo_tu is not None and loader_tu is not None:
    dfs.append(evaluar_modelo_completo(
        modelo_tu, loader_tu, "TransUNet",
        modo="custom", es_segmentacion=True, n_clases=9))

if modelo_swu is not None and loader_swu is not None:
    dfs.append(evaluar_modelo_completo(
        modelo_swu, loader_swu, "SwinUnet",
        modo="custom", es_segmentacion=True, n_clases=4))

if dfs:
    tabla_final = pd.concat(dfs, ignore_index=True)
    tabla_final.to_csv("/content/resultados_globales.csv", index=False)
    print(tabla_final)
else:
    print("⚠️ Ningún modelo entrenado. Ejecuta antes la celda de descarga "
          "y luego las celdas de cada transformer.")